# Connectar-se a l'ERP de Distribucions Es Pla

**Distribucions Es Pla, SL** reparteix menjar i begudes a hotels, restaurants i bars de Mallorca. Les comandes, els clients i els
productes són al seu ERP, una base de dades PostgreSQL que l'empresa té a Azure.

Abans d'extreure res, cal arribar-hi: saber on és, amb quin connector s'hi parla, com s'identifica qui s'hi connecta, i què hi ha a
dins sense tenir cap documentació. Aquest notebook fa aquests quatre passos.

Aquest notebook s'executa a JupyterHub de l'institut: el servidor de l'ERP només accepta connexions des de la xarxa de l'institut.

Documentació de referència:
- psycopg (el connector de PostgreSQL per a Python): https://www.psycopg.org/psycopg3/docs/
- Autenticació amb Microsoft Entra a PostgreSQL d'Azure: https://learn.microsoft.com/azure/postgresql/security/security-entra-configure
- El catàleg `information_schema`: https://www.postgresql.org/docs/16/information-schema.html

## 1. Les dades de la connexió

Per connectar-se a qualsevol base de dades calen les mateixes peces. L'administrador de l'ERP ens les ha passat així:

In [ ]:
HOST = "pg-5104-distribucio.postgres.database.azure.com"
PORT = 5432
BASE_DE_DADES = "erp_distribucio"
SSLMODE = "require"          # el servidor només accepta connexions xifrades
TENANT = "politecnicllevant.cat"   # el directori Entra de l'institut

No hi ha cap contrasenya. L'ERP fa servir **Microsoft Entra ID**: cadascú hi entra amb el seu compte de l'institut, i el que fa de
contrasenya és un **token** temporal que dona Microsoft després d'iniciar la sessió.

Escriviu el vostre correu de l'institut a la cel·la següent.

In [ ]:
USUARI = "nom@alumnes.politecnicllevant.cat"   # el vostre correu de l'institut

## 2. Obtenir el token i connectar-se

La cel·la següent demana el token. Surt un missatge amb una adreça i un codi: s'obre l'adreça en una altra pestanya, s'hi escriu el
codi i s'inicia la sessió amb el compte de l'institut. El token dura aproximadament una hora; quan caduca, es torna a executar
aquesta cel·la.

In [ ]:
import os

import psycopg
from azure.identity import DeviceCodeCredential

ESCOP = "https://ossrdbms-aad.database.windows.net/.default"   # el "recurs" de les bases de dades d'Azure


def obtenir_token():
    if os.environ.get("ERP_TOKEN"):          # només per a proves del professor
        return os.environ["ERP_TOKEN"]
    credencial = DeviceCodeCredential(tenant_id=TENANT)
    return credencial.get_token(ESCOP).token


USUARI = os.environ.get("ERP_USER", USUARI)
TOKEN = obtenir_token()


def connecta(base_de_dades=BASE_DE_DADES):
    return psycopg.connect(host=HOST, port=PORT, dbname=base_de_dades, user=USUARI,
                           password=TOKEN, sslmode=SSLMODE, connect_timeout=15)


con = connecta()
print("Connectat com a", con.execute("SELECT current_user").fetchone()[0])
print(con.execute("SELECT version()").fetchone()[0])
print("Connexió xifrada:", con.info.pgconn.ssl_in_use)

Fixeu-vos en el que acaba de passar:

- `psycopg` és el **connector**: la biblioteca que sap parlar el protocol de PostgreSQL.
- Les dades de la connexió (servidor, port, base de dades, usuari, xifratge) són la **cadena de connexió**.
- El token ha fet de contrasenya: no n'hi ha cap d'escrita al notebook.

## 3. Què hi ha a dins?

Ningú no us ha donat el model de dades. Cada base de dades relacional el té escrit en el seu propi **catàleg**: `information_schema`
és l'estàndard SQL, i PostgreSQL té també `pg_catalog`.

Exemple guiat: les taules de l'esquema `erp`.

In [ ]:
import pandas as pd


def consulta(sql, params=None):
    """Executa una consulta i torna un DataFrame amb els noms de les columnes."""
    with con.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


consulta("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'erp'
    ORDER BY table_name
""")

### TODO 1: quantes files té cada taula?

En un sistema origen no es fa `SELECT count(*)` de taules grans: obliga a llegir-les senceres, i l'ERP l'està fent servir l'empresa.
PostgreSQL guarda una **estimació** a `pg_class.reltuples`, que s'actualitza quan s'analitza la taula.

Escriviu una consulta que torni, per a cada taula de l'esquema `erp`, el nom i les files estimades, de més a menys.
Pista: `pg_class` (columnes `relname`, `reltuples`, `relkind = 'r'`) i `pg_namespace` (columna `nspname`), unides per
`pg_class.relnamespace = pg_namespace.oid`.

In [ ]:
raise NotImplementedError("TODO 1: files estimades de cada taula de l'esquema erp")

### TODO 2: les columnes de les comandes

Escriviu una consulta a `information_schema.columns` que torni el nom, el tipus de dada i si admet nuls de les columnes de la taula
`comandes`, en l'ordre en què estan definides (`ordinal_position`).

In [ ]:
raise NotImplementedError("TODO 2: columnes de la taula comandes")

### TODO 3: com es relacionen les taules?

Les claus foranes diuen quines taules s'uneixen amb quines. La consulta ja està escrita: executeu-la i, a la cel·la de text de sota,
dibuixeu el model (en text, per exemple `comandes.client_id -> clients.id`).

In [ ]:
consulta("""
    SELECT tc.table_name  AS taula,
           kcu.column_name AS columna,
           ccu.table_name  AS taula_referenciada,
           ccu.column_name AS columna_referenciada
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
      ON kcu.constraint_name = tc.constraint_name AND kcu.table_schema = tc.table_schema
    JOIN information_schema.constraint_column_usage ccu
      ON ccu.constraint_name = tc.constraint_name AND ccu.table_schema = tc.table_schema
    WHERE tc.constraint_type = 'FOREIGN KEY' AND tc.table_schema = 'erp'
    ORDER BY 1, 2
""")

*El model, en text:*

...

## 4. La primera extracció

La gerent vol saber quantes comandes van entrar el **setembre de 2026** per **canal** (comercial, telèfon, web, app), sense
comptar les anul·lades.

Exemple guiat: les comandes d'un sol dia, amb un **paràmetre** (`%s`) en lloc d'escriure la data dins del text de la consulta.
Així el connector s'encarrega de les cometes i del format, i no hi ha risc d'injecció SQL.

In [ ]:
consulta("""
    SELECT num_comanda, data_comanda, estat, canal
    FROM erp.comandes
    WHERE data_comanda >= %s AND data_comanda < %s
    ORDER BY data_comanda
    LIMIT 10
""", ("2026-09-15", "2026-09-16"))

### TODO 4: comandes del setembre de 2026 per canal

Amb una sola consulta parametritzada: comandes del setembre de 2026, sense les anul·lades (`estat <> 'ANUL·LADA'`), agrupades per
`canal`, de més a menys. Guardeu el resultat a la variable `per_canal`.

In [ ]:
raise NotImplementedError("TODO 4: comandes del setembre de 2026 per canal")

## 5. Quan la connexió falla

Cada error diu a quina **capa** s'ha aturat la connexió. Executeu les cel·les i llegiu el missatge.

In [ ]:
try:
    connecta("erp")
except psycopg.OperationalError as e:
    print(e)

In [ ]:
try:
    con.execute("DELETE FROM erp.comandes WHERE id = -1")
except psycopg.Error as e:
    print(type(e).__name__, ":", e)
    con.rollback()

### TODO 5: diagnosticar

Per a cada missatge, escriviu a quina capa s'ha aturat (nom del servidor, xarxa i tallafoc, xifratge, identitat, permisos o objecte
de la base de dades) i què faríeu:

1. `failed to resolve host 'pg-5104-distribucion.postgres.database.azure.com'`
2. `connection timeout expired`
3. `no pg_hba.conf entry for host "83.49.226.166", user "...", database "erp_distribucio", no encryption`
4. `The access token has invalid format.`
5. `database "erp" does not exist`
6. `cannot execute DELETE in a read-only transaction`
7. `relation "erp.factures" does not exist`

*Respostes:*

1. ...

## 6. Tancar

Una connexió oberta ocupa recursos al servidor. En acabar, es tanca.

In [ ]:
con.close()